# MERIT-UNETR++: Colab connection and GPU preflight

In Colab choose **Runtime > Change runtime type > GPU**, connect, then run the cells in order. This notebook installs dependencies, checks the saved split, and optionally profiles a complete synthetic optimizer step. It does not start training or select a normalization policy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

project = Path('/content/drive/MyDrive/PROJECT HCMUS/Paper/merit-covid-colab-2026-11-08')
assert (project / 'pyproject.toml').is_file(), f'Project not found: {project}'
assert (project / 'data/covid19_20/cases.json').is_file(), 'Dataset manifest missing'
os.chdir(project)
print('Project:', Path.cwd())
print('CT and mask files:', len(list((project / 'data/covid19_20/Train').glob('*.nii.gz'))))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[unetrpp]'], check=True)

In [ ]:
# Query the GPU in a short-lived process so the notebook kernel does not
# retain a CUDA context while the profiler measures its own memory.
probe = subprocess.run([sys.executable, '-c',
    "import torch; print('PyTorch:', torch.__version__); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"], check=True)

In [ ]:
# Read saved split metadata and verify the expected image/mask files exist.
subprocess.run([sys.executable, 'scripts/merit_pipeline.py', 'check-split',
                '--config', 'configs/merit_index.yaml'], check=True)

## Optional: GPU memory check
The next cell tries 256x256, batch size 1, with synthetic data and a full optimizer step. An out-of-memory result is a valid finding. It does not test real CTs or train a research model. Outputs use a new temporary folder each run.

In [ ]:
import json, tempfile
gpu_check = subprocess.run([sys.executable, '-c', 'import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)'])
if gpu_check.returncode != 0:
    raise RuntimeError('No GPU detected; connect a GPU runtime first.')
profile_path = Path(tempfile.mkdtemp(prefix='merit_gpu_')) / 'profile.json'
result = subprocess.run([sys.executable, 'scripts/merit_preflight.py', 'profile',
    '--config', 'configs/merit_index.yaml', '--size', '256', '--batch-size', '1',
    '--device', 'cuda', '--output', str(profile_path)])
if profile_path.exists():
    report = json.loads(profile_path.read_text())
    print('Status:', report['status'])
    print('Peak allocated MiB:', report.get('peak_allocated_mib'))
    print('Mean step seconds:', report.get('mean_step_seconds'))
    if report.get('error'):
        print('Error:', report['error'])
else:
    raise RuntimeError(f'Profile failed before producing a report (exit {result.returncode})')

The production experiment currently uses native intensity and 128x128. The normalization proposal is not implemented yet; changing to 256 requires a matching experiment config and a real-CT pilot. For later training, write checkpoints to Drive and retain the same resolved dataset paths when resuming.